# Pairwise Experiments

### Setup

In [ ]:
# You can set them inline
# import os
# os.environ["OPENAI_API_KEY"] = ""
# os.environ["LANGSMITH_API_KEY"] = ""
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_PROJECT"] = "langsmith-notebook"

In [1]:
# Or you can use a .env file
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

True

### Task

我们来设置一个新任务！这里有一位名叫 Bob 汽车公司销售员。Bob 有很多订单，所以他想根据一些会议记录总结这些订单的情况。

Bob 正在尝试几种不同的提示词，以便为他的交易生成简洁明了的文字记录。

Bob 整理了一份包含他所有交易记录的数据集，我们这就把它加载进来。请注意，这并非最终的黄金数据集，这里没有参考输出。

In [2]:
from langsmith import Client

client = Client()
dataset = client.clone_public_dataset(
  "https://smith.langchain.com/public/9078d2f1-7bef-4ba7-b795-210a17682ef9/d"
)

### Experiments

现在，让我们使用两种不同的提示词在这个数据集上进行一些实验。我们再添加一个评估器，用来评价我们生成的摘要质量！

In [5]:
import os
from pydantic import BaseModel, Field
from openai import OpenAI

openai_client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1"
)

SUMMARIZATION_SYSTEM_PROMPT = """You are a judge, aiming to score how well a summary summarizes the content of a transcript"""

SUMMARIZATION_HUMAN_PROMPT = """
[The Meeting Transcript] {transcript}
[The Start of Summarization] {summary} [The End of Summarization]
Output your response in JSON format with the key 'score'."""

class SummarizationScore(BaseModel):
    score: int = Field(description="""A score from 1-5 ranking how good the summarization is for the provided transcript, with 1 being a bad summary, and 5 being a great summary""")
    
def summary_score_evaluator(inputs: dict, outputs: dict) -> list:
    completion = openai_client.beta.chat.completions.parse(
        model="qwen3-max",
        messages=[
            {   
                "role": "system",
                "content": SUMMARIZATION_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": SUMMARIZATION_HUMAN_PROMPT.format(
                    transcript=inputs["transcript"],
                    summary=outputs.get("output", "N/A"),
                )}
        ],
        response_format=SummarizationScore,
    )

    summary_score = completion.choices[0].message.parsed.score
    return {"key": "summary_score", "score": summary_score}

首先，我们将使用一个较好的提示词版本来运行我们的实验！

In [6]:
# Prompt One: Good Prompt!
def good_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="qwen3-max",
        messages=[
            {
                "role": "user",
                "content": f"Concisely summarize this meeting in 3 sentences. Make sure to include all of the important events. Meeting: {inputs['transcript']}"
            }
        ],
    )
    return response.choices[0].message.content

client.evaluate(
    good_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="Good Summarizer"
)

View the evaluation results for experiment: 'Good Summarizer-d14b9442' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/45f8dbce-6b08-4401-be39-a41e97dc1e18/compare?selectedSessions=971f26d0-9cfc-4abf-bd65-720c95927c68




5it [00:28,  5.78s/it]


,inputs.transcript,outputs.output,error,feedback.summary_score,execution_time,example_id,id
0,"Bob and Mr. Patel (CLOSED DEAL): Bob: Hello, M...",Bob assisted Mr. Patel in selecting a midsize ...,None,10,3.848614,01f8f8b8-ad61-4a1f-869d-10f6b9250d1e,019b1207-d65f-70c8-bd1b-56e5c1899626
1,Bob and Mr. Carter (CLOSED DEAL): Bob: Welcome...,Mr. Carter visited Bob to trade in his 2015 Ta...,None,10,10.045356,1de7e1a2-8ca3-4fb2-8b0c-284116c2af00,019b1207-e86c-724f-b6ad-2fc6a3f3de5d
2,Bob and Mr. Johnson (CLOSED DEAL): Bob: Good m...,"Bob welcomed Mr. Johnson to Ford Motors, where...",None,5,3.516351,9f96a898-0ff5-45e0-bc82-eaa83a7a524f,019b1208-1282-75e6-a4bb-a2dde42d6eca
3,Bob and Ms. Nguyen (NO DEAL): Bob: Good aftern...,Bob greeted Ms. Nguyen and recommended the For...,None,5,3.439901,be4fe7b3-b30a-4a34-94e9-36cff4148432,019b1208-2377-7294-866e-f75d32ee6765
4,"Bob and Ms. Thompson (NO DEAL): Bob: Hi, Ms. T...","Bob greeted Ms. Thompson at Ford Motors, where...",None,5,2.676851,d5fb17f8-bda4-4d4f-8dd6-3a0ade75826a,019b1208-340b-7358-aa5c-a00418474b11


现在，我们将使用一个更糟糕的提示版本进行实验，以突出两者之间的差异。

In [7]:
# Prompt Two: Worse Prompt!
def bad_summarizer(inputs: dict):
    response = openai_client.chat.completions.create(
        model="qwen3-max",
        messages=[
            {
                "role": "user",
                "content": f"Summarize this in one sentence. {inputs['transcript']}"
            }
        ],
    )
    return response.choices[0].message.content

client.evaluate(
    bad_summarizer,
    data=dataset,
    evaluators=[summary_score_evaluator],
    experiment_prefix="Bad Summarizer"
)

View the evaluation results for experiment: 'Bad Summarizer-eaa52c8c' at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/45f8dbce-6b08-4401-be39-a41e97dc1e18/compare?selectedSessions=202c94cb-7ba8-4165-87ad-36aa0914414d




5it [00:20,  4.07s/it]


,inputs.transcript,outputs.output,error,feedback.summary_score,execution_time,example_id,id
0,"Bob and Mr. Patel (CLOSED DEAL): Bob: Hello, M...","Bob successfully closed a deal with Mr. Patel,...",None,5,1.877764,01f8f8b8-ad61-4a1f-869d-10f6b9250d1e,019b120a-f255-725b-af7c-1dca75ee2aee
1,Bob and Mr. Carter (CLOSED DEAL): Bob: Welcome...,Bob successfully closed a deal with Mr. Carter...,None,5,2.130885,1de7e1a2-8ca3-4fb2-8b0c-284116c2af00,019b120a-fdea-7461-827e-7b0f05544c9d
2,Bob and Mr. Johnson (CLOSED DEAL): Bob: Good m...,Bob successfully closed a deal with Mr. Johnso...,None,9,3.577490,9f96a898-0ff5-45e0-bc82-eaa83a7a524f,019b120b-0928-7500-b11d-e353474298a2
3,Bob and Ms. Nguyen (NO DEAL): Bob: Good aftern...,"Bob tried to sell Ms. Nguyen a Ford Fiesta, Mu...",None,5,2.083248,be4fe7b3-b30a-4a34-94e9-36cff4148432,019b120b-2010-73ed-b549-5ceb1c4e6ab8
4,"Bob and Ms. Thompson (NO DEAL): Bob: Hi, Ms. T...",Bob warmly assists Ms. Thompson as she browses...,None,5,4.278590,d5fb17f8-bda4-4d4f-8dd6-3a0ade75826a,019b120b-2b60-7169-89fc-7d270f0c4b52


### Pairwise Experiment

让我们定义一个用于比较两个实验的函数。成对评估函数可访问以下字段：
- `inputs: dict`: 数据集中单个示例对应的输入字典。
- `outputs: list[dict]`: 每个实验在给定输入上生成的字典输出列表。
- `reference_outputs: dict`: 与示例关联的参考输出字典（如有可用）。
- `runs: list[Run]`: 实验在给定示例上生成的完整运行对象列表。若需访问各运行步骤的中间结果或元数据，请使用此字段。
- `example: Example`: 完整的数据集示例，包含示例输入、输出（若有）及元数据（若有）。

首先，让我们给担任评判者的LLM下达指令。在本案例中，我们将直接使用担任评判者的LLM来评定哪个摘要生成器最具实用价值。

在缺乏真实数据参考的情况下，评判摘要生成器可能存在难度，但通过直接对比不同提示词的效果，我们仍能初步判断优劣！

In [11]:
JUDGE_SYSTEM_PROMPT = """
Please act as an impartial judge and evaluate the quality of the summarizations provided by two AI summarizers to the meeting transcript below.
Your evaluation should consider factors such as the helpfulness, relevance, accuracy, depth, creativity, and level of detail of their summarizations. 
Begin your evaluation by comparing the two summarizations and provide a short explanation. 
Avoid any position biases and ensure that the order in which the responses were presented does not influence your decision. 
Do not favor certain names of the assistants. 
Be as objective as possible. """

JUDGE_HUMAN_PROMPT = """
[The Meeting Transcript] {transcript}

[The Start of Assistant A's Summarization] {answer_a} [The End of Assistant A's Summarization]

[The Start of Assistant B's Summarization] {answer_b} [The End of Assistant B's Summarization]

Output your response in JSON format with the key 'preference' value must be an integer."""

函数`ranked_preference`将接收一个`inputs`字典，以及用于比较不同实验的`outputs`字典列表。

In [12]:
from pydantic import BaseModel, Field

class Preference(BaseModel):
    preference: int = Field(description="""1 if Assistant A answer is better based upon the factors above.
2 if Assistant B answer is better based upon the factors above.
Output 0 if it is a tie.""")
    
def ranked_preference(inputs: dict, outputs: list[dict]) -> list:
    completion = openai_client.beta.chat.completions.parse(
        model="qwen3-max",
        messages=[
            {   
                "role": "system",
                "content": JUDGE_SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": JUDGE_HUMAN_PROMPT.format(
                    transcript=inputs["transcript"],
                    answer_a=outputs[0].get("output", "N/A"),
                    answer_b=outputs[1].get("output", "N/A")
                )}
        ],
        response_format=Preference,
    )

    preference_score = completion.choices[0].message.parsed.preference

    if preference_score == 1:
        scores = [1, 0]
    elif preference_score == 2:
        scores = [0, 1]
    else:
        scores = [0, 0]
    return scores

现在让我们使用`evaluate()`运行我们的成对实验

In [13]:
from langsmith import evaluate

evaluate(
    ("Good Summarizer-d14b9442", "Bad Summarizer-eaa52c8c"),  # TODO: 替换为你的实验名称
    evaluators=[ranked_preference]
)

View the pairwise evaluation results at:
https://smith.langchain.com/o/e355c563-0e02-4eb4-baa7-8e9e5f87b8ff/datasets/45f8dbce-6b08-4401-be39-a41e97dc1e18/compare?selectedSessions=971f26d0-9cfc-4abf-bd65-720c95927c68%2C202c94cb-7ba8-4165-87ad-36aa0914414d&comparativeExperiment=ade75b21-bdfb-4c24-a405-70f6716b8767




100%|████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:07<00:00,  1.43s/it]
